In [269]:
import pandas as pd

df = pd.read_csv("../data/raw/IPL.csv")

print(df.shape)
df.head()

C:\Users\Shiva\AppData\Local\Temp\ipykernel_2712\2085303613.py:3: DtypeWarning: Columns (0: review_batter, 1: team_reviewed, 2: review_decision, 3: umpire, 4: season, 5: superover_winner, 6: result_type, 7: method, 8: event_match_no) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/IPL.csv")


(278205, 64)


,Unnamed: 0,match_id,date,match_type,event_name,innings,batting_team,bowling_team,over,ball,...,team_runs,team_balls,team_wicket,new_batter,batter_runs,batter_balls,bowler_wicket,batting_partners,next_batter,striker_out
0,131970,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,...,1,1,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
1,131971,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,...,1,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
2,131972,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,2,0,NaN,0,1,0,"('BB McCullum', 'SC Ganguly')",NaN,False
3,131973,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,...,2,3,0,NaN,0,2,0,"('BB McCullum', 'SC Ganguly')",NaN,False
4,131974,335982,2008-04-18,T20,Indian Premier League,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,...,2,4,0,NaN,0,3,0,"('BB McCullum', 'SC Ganguly')",NaN,False


In [270]:
df = df[[
    'match_id',
    'date',
    'batting_team',
    'bowling_team',
    'venue',
    'batter',
    'bowler',
    'runs_total',
    'wicket_kind'
]]

In [271]:
df.head()

,match_id,date,batting_team,bowling_team,venue,batter,bowler,runs_total,wicket_kind
0,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,SC Ganguly,P Kumar,1,NaN
1,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN
2,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,1,NaN
3,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN
4,335982,2008-04-18,Kolkata Knight Riders,Royal Challengers Bangalore,M Chinnaswamy Stadium,BB McCullum,P Kumar,0,NaN


In [272]:
df.isnull().sum()

match_id             0
date                 0
batting_team         0
bowling_team         0
venue                0
batter               0
bowler               0
runs_total           0
wicket_kind     264382
dtype: int64

In [273]:
df['wicket_kind'] = df['wicket_kind'].fillna('None')

In [274]:
df['date'] = pd.to_datetime(df['date'])

In [275]:
df['batting_team'] = df['batting_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

df['bowling_team'] = df['bowling_team'].replace({
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings'
})

In [276]:
team_scores = df.groupby(['match_id', 'batting_team'])['runs_total'].sum().reset_index()

In [277]:
team_scores.head()

,match_id,batting_team,runs_total
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Punjab Kings,207
4,335984,Delhi Capitals,132


In [278]:
team_scores.to_csv("../data/processed/team_scores.csv", index=False)

In [279]:
match_df = team_scores.pivot(index='match_id', columns='batting_team', values='runs_total').reset_index()

In [280]:
match_df.head()

batting_team,match_id,Chennai Super Kings,Deccan Chargers,Delhi Capitals,Gujarat Lions,Gujarat Titans,Kochi Tuskers Kerala,Kolkata Knight Riders,Lucknow Super Giants,Mumbai Indians,Pune Warriors,Punjab Kings,Rajasthan Royals,Rising Pune Supergiant,Rising Pune Supergiants,Royal Challengers Bangalore,Royal Challengers Bengaluru,Sunrisers Hyderabad
0,335982,NaN,NaN,NaN,NaN,NaN,NaN,222.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,82.0,NaN,NaN
1,335983,240.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,207.0,NaN,NaN,NaN,NaN,NaN,NaN
2,335984,NaN,NaN,132.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,129.0,NaN,NaN,NaN,NaN,NaN
3,335985,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,165.0,NaN,NaN,NaN,NaN,NaN,166.0,NaN,NaN
4,335986,NaN,110.0,NaN,NaN,NaN,NaN,112.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [281]:
def process_match(x):
    # Only keep matches with exactly 2 teams
    if len(x) != 2:
        return None
    
    team1 = x.iloc[0]['batting_team']
    team2 = x.iloc[1]['batting_team']
    
    score1 = x.iloc[0]['runs_total']
    score2 = x.iloc[1]['runs_total']
    
    winner = team1 if score1 > score2 else team2
    
    return pd.Series({
        'team1': team1,
        'team2': team2,
        'score1': score1,
        'score2': score2,
        'winner': winner
    })

match_df = team_scores.groupby('match_id').apply(process_match).dropna().reset_index()

In [282]:
match_df.head()

,match_id,team1,team2,score1,score2,winner
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,222.0,82.0,Kolkata Knight Riders
1,335983,Chennai Super Kings,Punjab Kings,240.0,207.0,Chennai Super Kings
2,335984,Delhi Capitals,Rajasthan Royals,132.0,129.0,Delhi Capitals
3,335985,Mumbai Indians,Royal Challengers Bangalore,165.0,166.0,Royal Challengers Bangalore
4,335986,Deccan Chargers,Kolkata Knight Riders,110.0,112.0,Kolkata Knight Riders


In [283]:
match_df.to_csv("../data/processed/match_results.csv", index=False)

In [284]:
#Creating players Runs per match

In [285]:
player_runs = df.groupby(['match_id', 'batter'])['runs_total'].sum().reset_index()

In [286]:
player_runs.head()

,match_id,batter,runs_total
0,335982,AA Noffke,11
1,335982,B Akhil,0
2,335982,BB McCullum,169
3,335982,CL White,6
4,335982,DJ Hussey,12


In [287]:
player_runs = player_runs.sort_values(by=['batter', 'match_id'])

In [288]:
player_runs['form'] = player_runs.groupby('batter')['runs_total'].rolling(5).mean().reset_index(0, drop=True)

In [289]:

player_runs.head(10)

,match_id,batter,runs_total,form
4299,548346,A Ashish Reddy,10,NaN
4390,548352,A Ashish Reddy,3,NaN
4496,548359,A Ashish Reddy,12,NaN
4699,548373,A Ashish Reddy,10,NaN
4747,548376,A Ashish Reddy,5,8.0
4866,598000,A Ashish Reddy,7,7.4
4933,598004,A Ashish Reddy,14,9.6
5027,598010,A Ashish Reddy,16,10.4
5076,598013,A Ashish Reddy,4,9.2
5160,598018,A Ashish Reddy,19,12.0


In [290]:
player_runs['form'] = player_runs['form'].fillna(0)

In [291]:
player_runs.to_csv("../data/processed/player_form.csv", index=False)

#SECTION 5 Feature Engineering

In [292]:
df = df.merge(player_runs[['match_id', 'batter', 'form']], 
              on=['match_id', 'batter'], 
              how='left')

In [293]:
team_form = df.groupby(['match_id', 'batting_team'])['form'].mean().reset_index()

In [294]:
final_df = match_df.merge(team_form, 
                          left_on=['match_id', 'team1'], 
                          right_on=['match_id', 'batting_team'], 
                          how='left')

final_df = final_df.rename(columns={'form': 'team1_form'}).drop('batting_team', axis=1)

final_df = final_df.merge(team_form, 
                          left_on=['match_id', 'team2'], 
                          right_on=['match_id', 'batting_team'], 
                          how='left')

final_df = final_df.rename(columns={'form': 'team2_form'}).drop('batting_team', axis=1)

In [295]:
final_df.head(40)

,match_id,team1,team2,score1,score2,winner,team1_form,team2_form
0,335982,Kolkata Knight Riders,Royal Challengers Bangalore,222.0,82.0,Kolkata Knight Riders,0.000000,0.000000
1,335983,Chennai Super Kings,Punjab Kings,240.0,207.0,Chennai Super Kings,0.000000,0.000000
2,335984,Delhi Capitals,Rajasthan Royals,132.0,129.0,Delhi Capitals,0.000000,0.000000
3,335985,Mumbai Indians,Royal Challengers Bangalore,165.0,166.0,Royal Challengers Bangalore,0.000000,0.000000
4,335986,Deccan Chargers,Kolkata Knight Riders,110.0,112.0,Kolkata Knight Riders,0.000000,0.000000
5,335987,Punjab Kings,Rajasthan Royals,166.0,168.0,Rajasthan Royals,0.000000,0.000000
6,335988,Deccan Chargers,Delhi Capitals,142.0,143.0,Delhi Capitals,0.000000,0.000000
7,335989,Chennai Super Kings,Mumbai Indians,208.0,202.0,Chennai Super Kings,0.000000,0.000000
8,335990,Deccan Chargers,Rajasthan Royals,214.0,217.0,Rajasthan Royals,0.000000,0.000000
9,335991,Mumbai Indians,Punjab Kings,116.0,182.0,Punjab Kings,0.000000,0.000000


In [296]:
# Venue feature
venue_df = df[['match_id', 'venue']].drop_duplicates()

final_df = final_df.merge(venue_df, on='match_id', how='left')

final_df = pd.get_dummies(final_df, columns=['venue'])

In [297]:
#new feature toss diff
# Toss feature
toss_df = df[['match_id', 'batting_team']].drop_duplicates()

final_df = final_df.merge(toss_df, on='match_id', how='left')

final_df['team1_batting'] = (final_df['team1'] == final_df['batting_team']).astype(int)

In [298]:
# New feature form diff
final_df['form_diff'] = final_df['team1_form'] - final_df['team2_form']

In [299]:
final_df['target'] = (final_df['winner'] == final_df['team1']).astype(int)

In [300]:
model_df = final_df.drop(columns=['match_id', 'team1', 'team2', 'winner', 'batting_team'])

In [301]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

final_df = final_df.sort_values(by='match_id')

model_df = final_df[[
    'team1_form',
    'team2_form',
    'form_diff',
    'team1_batting',
    'target'
]]

X = model_df.drop('target', axis=1)
y = model_df['target']

split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05
)
model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [302]:
y_pred = model.predict(X_test)

In [303]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.6566523605150214


#SHAP ANALYSIS

In [ ]:
import shap

explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)

import matplotlib.pyplot as plt


ImportError: matplotlib is not installed so plotting is not available! Run `pip install matplotlib` to fix this.